## ÉTAPE 10 — Nettoyage avancé des données
Objectif : gérer valeurs manquantes, doublons, outliers


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("JEU_DE_DONNEE.csv")


## 1. VALEURS MANQUANTES


In [ ]:
print("--- Valeurs manquantes ---")
print(df.isnull().sum())
print(f"\nPourcentage manquant dans 'value' : {df['value'].isnull().mean()*100:.1f}%")

# Supprimer les lignes sans valeur
df_clean = df.dropna(subset=["value"])
print(f"\nLignes avant : {len(df):,} | Après dropna : {len(df_clean):,}")

# Remplir les manquants par la médiane (alternative)
df_fill = df.copy()
df_fill["value"] = df_fill["value"].fillna(df_fill["value"].median())


## 2. DOUBLONS


In [ ]:
doublons = df.duplicated().sum()
print(f"\nNombre de doublons : {doublons}")

# Supprimer les doublons
df_clean = df_clean.drop_duplicates()
print(f"Après suppression doublons : {len(df_clean):,} lignes")


## 3. DÉTECTER LES OUTLIERS — MÉTHODE IQR


## IQR = écart interquartile (Q3 - Q1)
Un outlier est une valeur en dehors de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]


In [ ]:
valeurs = df_clean["value"].dropna()

Q1  = valeurs.quantile(0.25)
Q3  = valeurs.quantile(0.75)
IQR = Q3 - Q1

borne_basse = Q1 - 1.5 * IQR
borne_haute = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean["value"] < borne_basse) | (df_clean["value"] > borne_haute)]
print(f"\n--- Méthode IQR ---")
print(f"Q1={Q1:.0f}, Q3={Q3:.0f}, IQR={IQR:.0f}")
print(f"Bornes : [{borne_basse:.0f} ; {borne_haute:.0f}]")
print(f"Outliers détectés : {len(outliers):,} ({len(outliers)/len(df_clean)*100:.1f}%)")


## 4. DÉTECTER LES OUTLIERS — MÉTHODE Z-SCORE


## Z-score = nombre d'écarts-types par rapport à la moyenne
On considère outlier tout Z > 3 ou Z < -3


In [ ]:
from scipy import stats
df_clean = df_clean.copy()
df_clean["zscore"] = np.abs(stats.zscore(df_clean["value"].fillna(0)))

outliers_z = df_clean[df_clean["zscore"] > 3]
print(f"\n--- Méthode Z-score (|z| > 3) ---")
print(f"Outliers détectés : {len(outliers_z):,}")


## 5. GRAPHIQUE — BOXPLOT


In [ ]:
cultures_box = ["wheat", "maize", "rice_paddy", "sugar_cane"]
df_box = df_clean[
    (df_clean["category"].isin(cultures_box)) &
    (df_clean["element"] == "Production Quantity") &
    (df_clean["country_or_area"] == "World +")
]

fig, ax = plt.subplots(figsize=(10, 5))
data_plot = [df_box[df_box["category"] == c]["value"].dropna() / 1e6
             for c in cultures_box]

bp = ax.boxplot(data_plot, labels=cultures_box, patch_artist=True,
                medianprops=dict(color="red", linewidth=2))

colors = ["#93C5FD", "#86EFAC", "#FCD34D", "#F9A8D4"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

ax.set_title("Boxplot — Distribution de la production mondiale par culture",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Production (millions de tonnes)")
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig("boxplot_outliers.png", dpi=150, bbox_inches="tight")
print("\nBoxplot sauvegardé ✓")
plt.show()


## 6. RÉSUMÉ


In [ ]:
print("\n=== RÉSUMÉ NETTOYAGE ===")
print(f"Valeurs manquantes supprimées : {df['value'].isnull().sum():,}")
print(f"Doublons supprimés            : {doublons}")
print(f"Outliers IQR détectés         : {len(outliers):,}")
print(f"Outliers Z-score détectés     : {len(outliers_z):,}")
print("\nCommandes clés :")
print("  df.dropna()          → supprimer les lignes vides")
print("  df.fillna(valeur)    → remplacer les vides")
print("  df.drop_duplicates() → supprimer les doublons")
print("  IQR = Q3 - Q1        → détecter les valeurs extrêmes")
